In [ ]:

import os, json, time
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, log_loss, confusion_matrix
from sklearn.ensemble import RandomForestClassifier

CSV_PATH = Path("classification_with_c110_d110_errors_snr.csv")

OUT_DIR = Path.cwd() / "rf_c110_truncation_out"
RUNS_DIR = OUT_DIR / "runs"
REPORT_DIR = OUT_DIR / "analysis_report"
for d in [OUT_DIR, RUNS_DIR, REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

HEARTBEAT_PATH = OUT_DIR / "HEARTBEAT.json"
LAST_EVENT_PATH = OUT_DIR / "LAST_EVENT.txt"
MANIFEST_PATH = OUT_DIR / "RUNNING_MANIFEST.json"

def hb(payload):
    payload = dict(payload)
    payload["time"] = time.ctime()
    HEARTBEAT_PATH.write_text(json.dumps(payload, indent=2))

def log_event(msg):
    LAST_EVENT_PATH.write_text(f"[{time.ctime()}] {msg}\n")

# ------------------------
# Experiment config
# ------------------------
K_MIN, K_MAX = 1, 55
K_LIST = list(range(K_MIN, K_MAX + 1))

N_REPEATS = 100
TEST_FRAC = 0.15
VAL_FRAC = 0.15
VAL_FRAC_OF_REMAIN = VAL_FRAC / (1.0 - TEST_FRAC)

BASE_SEED = 1234
THR_GRID_SIZE = 800

RF_PARAMS = dict(
    n_estimators=500,
    max_features="sqrt",
    min_samples_leaf=2,
    min_samples_split=4,
    bootstrap=True,
    class_weight="balanced_subsample",
    n_jobs=-1,
    random_state=0,
)

manifest = dict(
    started_at=time.ctime(),
    csv=str(CSV_PATH),
    out_dir=str(OUT_DIR.resolve()),
    plan=dict(
        K_MIN=K_MIN, K_MAX=K_MAX, N_REPEATS=N_REPEATS,
        TEST_FRAC=TEST_FRAC, VAL_FRAC=VAL_FRAC,
        THR_GRID_SIZE=THR_GRID_SIZE,
        RF_PARAMS=RF_PARAMS,
    ),
)
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2))
log_event("Manifest created.")
hb({"status": "ready"})

print("OUT_DIR:", OUT_DIR.resolve())

OUT_DIR: /Users/kris/Desktop/Astrophysics/rf_c110_truncation_out


In [ ]:
df = pd.read_csv(CSV_PATH)
if "y" not in df.columns:
    raise KeyError("CSV must contain 'y'")

df["y"] = df["y"].astype(int)

C110 = [f"c{i:03d}" for i in range(110)]
missing = [c for c in C110 if c not in df.columns]
if missing:
    raise KeyError(f"Missing c110 columns, e.g. {missing[:6]}")

# c000..c054 = BP, c055..c109 = RP
BP = [f"c{i:03d}" for i in range(55)]
RP = [f"c{i:03d}" for i in range(55, 110)]

Xbp = df[BP].to_numpy(np.float32)
Xrp = df[RP].to_numpy(np.float32)
y_all = df["y"].to_numpy(int)

print("Xbp:", Xbp.shape, "Xrp:", Xrp.shape, "pos_rate:", float(y_all.mean()))

Xbp: (2815, 55) Xrp: (2815, 55) pos_rate: 0.19822380106571935


In [ ]:
def make_splits(y, n_repeats, base_seed):
    splitter = StratifiedShuffleSplit(n_splits=n_repeats, test_size=TEST_FRAC, random_state=base_seed)
    idx_all = np.arange(len(y))
    splits = []
    for r, (trainval_rel, test_rel) in enumerate(splitter.split(np.zeros(len(y)), y)):
        idx_trainval = idx_all[trainval_rel]
        idx_test = idx_all[test_rel]

        y_tv = y[idx_trainval]
        splitter2 = StratifiedShuffleSplit(n_splits=1, test_size=VAL_FRAC_OF_REMAIN, random_state=base_seed + 1000 + r)
        tr_rel2, va_rel2 = next(splitter2.split(np.zeros(len(y_tv)), y_tv))

        idx_train = idx_trainval[tr_rel2]
        idx_val   = idx_trainval[va_rel2]
        splits.append(dict(repeat=r, train_idx=idx_train, val_idx=idx_val, test_idx=idx_test))
    return splits

splits = make_splits(y_all, N_REPEATS, BASE_SEED)
print("splits:", len(splits), "sizes:", {k: len(splits[0][k]) for k in ["train_idx","val_idx","test_idx"]})

splits: 100 sizes: {'train_idx': 1969, 'val_idx': 423, 'test_idx': 423}


In [ ]:
def make_X_for_K(idx, K):
    bp = Xbp[idx, :K]   # (N,K)
    rp = Xrp[idx, :K]   # (N,K)
    X = np.concatenate([bp, rp], axis=1).astype(np.float32, copy=False)  # (N, 2K)
    return X

# quick check
Xdemo = make_X_for_K(np.arange(5), 10)
print("demo shape:", Xdemo.shape)

demo shape: (5, 20)


In [ ]:
def prob_metrics(y_true, p):
    y_true = np.asarray(y_true).astype(int)
    p = np.asarray(p).astype(float)
    pc = np.clip(p, 1e-15, 1-1e-15)
    return dict(
        ROC_AUC=roc_auc_score(y_true, p) if len(np.unique(y_true)) > 1 else np.nan,
        PR_AUC=average_precision_score(y_true, p) if len(np.unique(y_true)) > 1 else np.nan,
        Brier=brier_score_loss(y_true, p),
        LogLoss=log_loss(y_true, pc, labels=[0,1]),
    )

def confusion_metrics(y_true, y_hat):
    tn, fp, fn, tp = confusion_matrix(y_true, y_hat, labels=[0,1]).ravel()
    sens = tp/(tp+fn) if (tp+fn) else np.nan
    spec = tn/(tn+fp) if (tn+fp) else np.nan
    prec = tp/(tp+fp) if (tp+fp) else np.nan
    return prec, sens, spec, tn, fp, fn, tp

def threshold_grid(p, n=800):
    lo, hi = float(np.min(p)), float(np.max(p))
    if not np.isfinite(lo) or not np.isfinite(hi) or lo == hi:
        return np.array([0.5])
    return np.linspace(lo, hi, n)

def choose_threshold(y_true, p, criterion="youden", n=800):
    best_thr, best_score = 0.5, -np.inf
    for thr in threshold_grid(p, n):
        yhat = (p >= thr).astype(int)
        prec, sens, spec, *_ = confusion_metrics(y_true, yhat)
        if criterion == "youden":
            score = (sens + spec - 1) if np.isfinite(sens) and np.isfinite(spec) else -np.inf
        elif criterion == "f1":
            score = (2*prec*sens/(prec+sens)) if np.isfinite(prec) and np.isfinite(sens) and (prec+sens)>0 else -np.inf
        else:
            raise ValueError
        if score > best_score:
            best_score, best_thr = float(score), float(thr)
    return best_thr, float(best_score)

def evaluate(yv, pv, yt, pt):
    out = {f"test_{k}": float(v) for k,v in prob_metrics(yt, pt).items()}
    for crit in ["youden", "f1"]:
        thr, sc = choose_threshold(yv, pv, crit, THR_GRID_SIZE)
        yhat = (pt >= thr).astype(int)
        prec, sens, spec, tn, fp, fn, tp = confusion_metrics(yt, yhat)
        out[f"{crit}_val_thr"] = float(thr)
        out[f"{crit}_val_score"] = float(sc)
        out[f"{crit}_test_Precision"] = float(prec)
        out[f"{crit}_test_Sensitivity"] = float(sens)
        out[f"{crit}_test_Specificity"] = float(spec)
        out[f"{crit}_test_tn"] = int(tn); out[f"{crit}_test_fp"] = int(fp)
        out[f"{crit}_test_fn"] = int(fn); out[f"{crit}_test_tp"] = int(tp)
    return out

In [ ]:
def run_id(K, rep):
    return f"K{int(K):02d}__r{int(rep):02d}"

def shard_path(K, rep):
    return RUNS_DIR / f"{run_id(K, rep)}.parquet"

def done(K, rep):
    return shard_path(K, rep).exists()

def save_row(row):
    pd.DataFrame([row]).to_parquet(RUNS_DIR / f"{row['run_id']}.parquet", index=False)

def load_all_rows():
    files = sorted(RUNS_DIR.glob("*.parquet"))
    if not files:
        return pd.DataFrame()
    return pd.concat([pd.read_parquet(p) for p in files], ignore_index=True)

print("Existing shards:", len(list(RUNS_DIR.glob("*.parquet"))))

Existing shards: 5500


In [ ]:
pK = tqdm(K_LIST, desc="K (coeffs per BP/RP)", leave=True)

for K in pK:
    pK.set_postfix_str(f"K={K}")
    log_event(f"Starting K={K}")

    p_rep = tqdm(splits, desc=f"Repeats [K={K}]", leave=False)
    for sp in p_rep:
        rep = sp["repeat"]
        if done(K, rep):
            continue

        hb({"status": "run_start", "K": int(K), "repeat": int(rep)})

        tr, va, te = sp["train_idx"], sp["val_idx"], sp["test_idx"]

        Xtr = make_X_for_K(tr, K); ytr = y_all[tr]
        Xva = make_X_for_K(va, K); yva = y_all[va]
        Xte = make_X_for_K(te, K); yte = y_all[te]

        # vary RF seed by repeat for a bit of diversity, but keep all else fixed
        rf_params = dict(RF_PARAMS)
        rf_params["random_state"] = BASE_SEED + rep

        model = RandomForestClassifier(**rf_params)

        t0 = time.time()
        model.fit(Xtr, ytr)

        pv = model.predict_proba(Xva)[:, 1]
        pt = model.predict_proba(Xte)[:, 1]

        met = evaluate(yva, pv, yte, pt)
        dt = time.time() - t0

        row = dict(
            run_id=run_id(K, rep),
            K=int(K),
            repeat=int(rep),
            runtime_s=float(dt),
            n_train=int(len(ytr)),
            n_val=int(len(yva)),
            n_test=int(len(yte)),
            test_pos_rate=float(np.mean(yte)),
            **met
        )
        save_row(row)

        hb({"status": "run_done", "K": int(K), "repeat": int(rep), "runtime_s": dt})

log_event("ALL DONE.")
hb({"status": "completed"})
print("Done. Shards:", len(list(RUNS_DIR.glob('*.parquet'))))

K (coeffs per BP/RP):   0%|          | 0/55 [00:00<?, ?it/s]

Repeats [K=1]:   0%|          | 0/100 [00:00<?, ?it/s]

Repeats [K=2]:   0%|          | 0/100 [00:00<?, ?it/s]

Repeats [K=3]:   0%|          | 0/100 [00:00<?, ?it/s]

Repeats [K=4]:   0%|          | 0/100 [00:00<?, ?it/s]

...

Repeats [K=52]:   0%|          | 0/100 [00:00<?, ?it/s]

Repeats [K=53]:   0%|          | 0/100 [00:00<?, ?it/s]

Repeats [K=54]:   0%|          | 0/100 [00:00<?, ?it/s]

Repeats [K=55]:   0%|          | 0/100 [00:00<?, ?it/s]

Done. Shards: 5500


In [ ]:
res = load_all_rows()
print("rows:", res.shape)
display(res.head(5))

res.to_parquet(OUT_DIR / "truncation_rf_raw.parquet", index=False)
res.to_csv(OUT_DIR / "truncation_rf_raw.csv", index=False)
print("Saved raw outputs.")

ArrowInvalid: Could not open Parquet input source '<Buffer>': Parquet magic bytes not found in footer. Either the file is corrupted or this is not a parquet file.

In [ ]:
metrics = [
    "runtime_s",
    "test_PR_AUC","test_ROC_AUC","test_LogLoss","test_Brier",
    "youden_test_Precision","youden_test_Sensitivity","youden_test_Specificity","youden_val_thr",
    "f1_test_Precision","f1_test_Sensitivity","f1_test_Specificity","f1_val_thr"
]

summ = res.groupby("K", as_index=False)[metrics].agg(["mean","std"])
summ.columns = [c if s == "" else f"{c}_{s}" for c, s in summ.columns]
summ = summ.sort_values("K")

summ.to_csv(OUT_DIR / "truncation_rf_summary_byK.csv", index=False)
display(summ.head(10))

,K,runtime_s_mean,runtime_s_std,test_PR_AUC_mean,test_PR_AUC_std,test_ROC_AUC_mean,test_ROC_AUC_std,test_LogLoss_mean,test_LogLoss_std,test_Brier_mean,...,youden_val_thr_mean,youden_val_thr_std,f1_test_Precision_mean,f1_test_Precision_std,f1_test_Sensitivity_mean,f1_test_Sensitivity_std,f1_test_Specificity_mean,f1_test_Specificity_std,f1_val_thr_mean,f1_val_thr_std
0,1,0.815910,0.023569,0.847101,0.034965,0.912923,0.019563,0.323713,0.089543,0.061696,...,0.435430,0.105167,0.881409,0.039931,0.772262,0.046698,0.973776,0.010218,0.566688,0.088190
1,2,0.820709,0.021916,0.863713,0.034198,0.923175,0.019115,0.287931,0.083382,0.055519,...,0.376083,0.087971,0.895989,0.041302,0.780000,0.048821,0.977050,0.010360,0.522140,0.097411
2,3,0.815061,0.014493,0.874281,0.032539,0.928883,0.019393,0.238749,0.054491,0.052123,...,0.369506,0.091280,0.890699,0.045980,0.798571,0.047639,0.974956,0.012844,0.475072,0.092003
3,4,0.815805,0.012698,0.877776,0.032054,0.930160,0.019651,0.220032,0.052092,0.050863,...,0.349571,0.078334,0.888325,0.042349,0.807262,0.051370,0.974159,0.011582,0.430332,0.083010
4,5,0.821676,0.010695,0.879454,0.033329,0.934075,0.019951,0.215572,0.041299,0.050657,...,0.337002,0.081805,0.889299,0.047180,0.801429,0.052040,0.974366,0.012784,0.431548,0.097701
5,6,0.821476,0.013639,0.886436,0.031208,0.935875,0.019354,0.208904,0.039515,0.050253,...,0.321870,0.090985,0.892515,0.040702,0.804762,0.049823,0.975280,0.011083,0.426821,0.073509
6,7,0.816690,0.015670,0.887413,0.030781,0.937227,0.019271,0.204334,0.034662,0.050321,...,0.314953,0.089453,0.891344,0.041874,0.803214,0.052521,0.974985,0.011340,0.417555,0.073287
7,8,0.826977,0.014919,0.887804,0.031553,0.938973,0.018519,0.210852,0.042973,0.050065,...,0.295227,0.081618,0.887161,0.044085,0.803333,0.054027,0.973805,0.012139,0.404592,0.091673
8,9,0.832394,0.014660,0.887276,0.032328,0.939201,0.018734,0.206861,0.040062,0.050271,...,0.299522,0.080395,0.888490,0.041702,0.802857,0.050517,0.974307,0.011454,0.402978,0.068353
9,10,0.832596,0.013779,0.888063,0.031480,0.939513,0.018425,0.204996,0.032873,0.050503,...,0.294343,0.077738,0.885932,0.044651,0.804048,0.049576,0.973569,0.012223,0.392913,0.071608


In [ ]:
import plotly.graph_objects as go
import plotly.express as px

def line_with_band(metric):
    dfm = res.groupby("K")[metric].agg(["mean","std"]).reset_index().sort_values("K")
    dfm.columns = ["K","mean","std"]
    dfm["lo"] = dfm["mean"] - dfm["std"]
    dfm["hi"] = dfm["mean"] + dfm["std"]

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=dfm["K"], y=dfm["mean"], mode="lines+markers", name=f"{metric} mean"))
    fig.add_trace(go.Scatter(x=dfm["K"], y=dfm["hi"], mode="lines", line=dict(width=0), showlegend=False))
    fig.add_trace(go.Scatter(x=dfm["K"], y=dfm["lo"], mode="lines", fill="tonexty",
                             line=dict(width=0), name="±1 std", opacity=0.2))
    fig.update_layout(title=f"{metric} vs K (mean ± std across repeats)",
                      xaxis_title="K", yaxis_title=metric, height=520)
    fig.show()

for m in [
    "test_PR_AUC","test_ROC_AUC","test_LogLoss","test_Brier",
    "youden_test_Sensitivity","youden_test_Specificity","youden_test_Precision",
    "f1_test_Sensitivity","f1_test_Specificity","f1_test_Precision"
]:
    line_with_band(m)